In [0]:
import pandas as pd
#import databricks.automl_runtime
import mlflow
import os
import uuid
import shutil
import pandas as pd
from sklearn.pipeline import Pipeline
#Necesita esto para ejecutarse el modelo 14.3.x-gpu-ml-scala2.12, and rerun it.

In [0]:
#import mlflow
#mlflow.set_registry_uri("databricks-uc")

In [0]:
## Schema para preparar modelos para desplegarlos
#%sql
#CREATE SCHEMA IF NOT EXISTS main.MLmodels;

##Test Modelo Oversampling x4 

In [0]:
CATALOG_NAME = "main"
SCHEMA_NAME = "default"

In [0]:
best_fraud_model = mlflow.pyfunc.load_model(
    'runs:/{run_id}/model'.format(
        run_id="168eb25129af48c4a927c1871f73e976"
    )
)

In [0]:
full_transactions = spark.read.format("delta").table("hive_metastore.default.business_full_transactions")

In [0]:
from pyspark.sql.functions import rand
from pyspark.sql.functions import col, desc

In [0]:
test_rows = full_transactions.limit(100)

In [0]:
yes_rows = full_transactions.orderBy(desc("IsFraud")).limit(100)

In [0]:
df_test = test_rows.unionByName(yes_rows)

In [0]:
y_true = df_test.select("IsFraud")
X_test = df_test.drop("IsFraud")

In [0]:
y_true.count()

1100

In [0]:
type(X_test)
X_testPandas = X_test.toPandas()

In [0]:
type(X_testPandas)

pandas.core.frame.DataFrame

In [0]:
predicciones = X_testPandas

In [0]:
predicciones["predictions"] = best_fraud_model.predict(X_testPandas)

[LightGBM] [Warning] lambda_l2 is set=3.553189557308358, reg_lambda=0.0 will be ignored. Current value: lambda_l2=3.553189557308358
[LightGBM] [Warning] lambda_l1 is set=0.1389447803458381, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.1389447803458381


In [0]:
y_truePandas = y_true.toPandas()

In [0]:
y_truePandas.head()

,IsFraud
0,No
1,No
2,No
3,No
4,No


In [0]:
predicciones["predictions"].head()

0    No
1    No
2    No
3    No
4    No
Name: predictions, dtype: object

In [0]:
df_compara = pd.DataFrame({
    "y_true": y_truePandas["IsFraud"],
    "y_pred": predicciones["predictions"]

})

In [0]:
df_compara.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1100 entries, 0 to 1099
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   y_true  1100 non-null   object
 1   y_pred  1100 non-null   object
dtypes: object(2)
memory usage: 17.3+ KB


In [0]:
df_compara["Aciertos"] = df_compara["y_true"] == df_compara["y_pred"]
df_compara["Aciertos"].value_counts()

True     1051
False      49
Name: Aciertos, dtype: int64

In [0]:
(49*100)/1100

4.454545454545454

In [0]:
df_compara.display()

y_true,y_pred,Aciertos
No,No,true
No,No,true
No,No,true
No,No,true
No,No,true
No,No,true
No,No,true
No,No,true
No,No,true
No,No,true


## API Python

In [0]:
import os
import requests
import numpy as np
import pandas as pd
import json

In [0]:
#Databricks Token

"----"

In [0]:
TestPandasDF = test_rows.toPandas()
TestPandasDF.head()

,User,Card,Amount,UseChip,MerchantName,MerchantCity,MerchantState,Zip,MCC,Errors,IsFraud,RangoHoras,DiaDeSemana,CardBrand,CardType,CardNumber,Expires,ExpiresMonth,ExpiresYear,CVV,HasChip,CardsIssued,CreditLimit,AcctOpenDate,AcctOpenMonth,AcctOpenYear,YearPINlastChanged,CardOnDarkWeb,Person,CurrentAge,RetirementAge,BirthYear,BirthMonth,Gender,Address,Apartment,City,State,Zipcode,Latitude,Longitude,PerCapitaIncomeZipcode,YearlyIncomePerson,TotalDebt,FICOScore,NumCreditCards
0,0,0,134.089996,Swipe Transaction,3527213246127876953,La Verne,CA,91750,5300,0,No,Madrugada,Sunday,Visa,Debit,4344676511950444,2022-12-01,12,2022,623,True,2,24295,2002-09-01,9,2002,2008,False,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,0,La Verne,CA,91750,34.150002,-117.760002,29278,59696,127613,787,5
1,0,0,38.480000,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754,5411,0,No,Madrugada,Sunday,Visa,Debit,4344676511950444,2022-12-01,12,2022,623,True,2,24295,2002-09-01,9,2002,2008,False,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,0,La Verne,CA,91750,34.150002,-117.760002,29278,59696,127613,787,5
2,0,0,120.339996,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754,5411,0,No,Madrugada,Monday,Visa,Debit,4344676511950444,2022-12-01,12,2022,623,True,2,24295,2002-09-01,9,2002,2008,False,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,0,La Verne,CA,91750,34.150002,-117.760002,29278,59696,127613,787,5
3,0,0,128.949997,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754,5651,0,No,Tarde,Monday,Visa,Debit,4344676511950444,2022-12-01,12,2022,623,True,2,24295,2002-09-01,9,2002,2008,False,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,0,La Verne,CA,91750,34.150002,-117.760002,29278,59696,127613,787,5
4,0,0,104.709999,Swipe Transaction,5817218446178736267,La Verne,CA,91750,5912,0,No,Madrugada,Tuesday,Visa,Debit,4344676511950444,2022-12-01,12,2022,623,True,2,24295,2002-09-01,9,2002,2008,False,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,0,La Verne,CA,91750,34.150002,-117.760002,29278,59696,127613,787,5


In [0]:
def create_tf_serving_json(data):
    return {'inputs': {name: data[name].tolist() for name in data.keys()} if isinstance(data, dict) else data.tolist()}

def score_model(dataset):
    url = 'https://----azuredatabricks.net/serving-endpoints/test/invocations'
    headers = {'Authorization': f'Bearer {token_databricks}', 'Content-Type': 'application/json'}

    ds_dict = {'dataframe_split': dataset.to_dict(orient='split')} if isinstance(dataset, pd.DataFrame) else create_tf_serving_json(dataset)

    data_json = json.dumps(ds_dict, default=str, allow_nan=True)
    response = requests.request(method='POST', headers=headers, url=url, data=data_json)
    if response.status_code != 200:
        raise Exception(f'Request failed with status {response.status_code}, {response.text}')
    return response.json()

    

In [0]:
respuesta = score_model(TestPandasDF)
print(respuesta)

{'predictions': ['No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No']}
